# Gurmukhi handwriting recognizer (free GPU)
This notebook downloads only the public IIIT Gurmukhi word corpus. Do not upload private documents. Select a free GPU accelerator, enable Internet, and run all cells. The final ZIP is enabled only when held-out source-script validation passes.


In [ ]:
%pip install -q 'accelerate>=1.1,<2' 'transformers>=4.45,<5' 'safetensors>=0.4,<1' 'sentencepiece>=0.2,<1'


In [ ]:
from pathlib import Path
training_source = '"""Fine-tune a source-language Gurmukhi TrOCR model on a free Kaggle GPU.\n\nThe official IIIT data is word-level.  This trainer retains word samples and\nalso composes deterministic multi-word lines so the exported checkpoint matches\nthe application\'s line-level HTR contract.  The artifact is never marked\nvalidated unless held-out CER, Unicode-script purity, and non-empty coverage\nall pass their configured gates.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport inspect\nimport json\nimport random\nimport shutil\nimport unicodedata\nimport urllib.request\nimport zipfile\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any, Iterable, Sequence\n\nfrom PIL import Image, ImageEnhance, ImageOps\n\n\nTRAINING_ARCHIVE_URL = (\n    "https://ilocr.iiit.ac.in/ihtr/assets/dataset/trainingset/gurumukhi.zip"\n)\nBASE_MODEL = "microsoft/trocr-small-stage1"\nGURMUKHI_START = 0x0A00\nGURMUKHI_END = 0x0A7F\n\n\n@dataclass(frozen=True, slots=True)\nclass Sample:\n    image_path: Path\n    text: str\n\n\ndef normalize_source_text(value: str) -> str:\n    """Normalize spacing and Unicode without changing source-language content."""\n    return " ".join(unicodedata.normalize("NFC", value).split())\n\n\ndef gurmukhi_ratio(value: str) -> float:\n    letters = [character for character in value if character.isalpha()]\n    if not letters:\n        return 0.0\n    return sum(GURMUKHI_START <= ord(character) <= GURMUKHI_END for character in letters) / len(\n        letters\n    )\n\n\ndef _within(child: Path, parent: Path) -> bool:\n    try:\n        child.resolve().relative_to(parent.resolve())\n        return True\n    except ValueError:\n        return False\n\n\ndef safe_extract(archive: Path, destination: Path) -> None:\n    """Extract a public dataset archive without permitting path traversal."""\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as source:\n        for member in source.infolist():\n            target = destination / member.filename\n            if not _within(target, destination):\n                raise ValueError(f"Unsafe archive member: {member.filename}")\n        source.extractall(destination)\n\n\ndef download_with_resume(url: str, destination: Path) -> Path:\n    """Download once; completed archives are reused across notebook reruns."""\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.is_file() and destination.stat().st_size > 0:\n        return destination\n    partial = destination.with_suffix(destination.suffix + ".partial")\n    request = urllib.request.Request(url, headers={"User-Agent": "OCRModelApp-HTR/1.0"})\n    with urllib.request.urlopen(request, timeout=120) as response, partial.open("wb") as output:\n        shutil.copyfileobj(response, output, length=1024 * 1024)\n    partial.replace(destination)\n    return destination\n\n\ndef _resolve_image(\n    relative_name: str,\n    label_file: Path,\n    data_root: Path,\n    basename_index: dict[str, Path] | None,\n) -> tuple[Path | None, dict[str, Path] | None]:\n    normalized = relative_name.replace("\\\\", "/").lstrip("./")\n    candidates = (label_file.parent / normalized, data_root / normalized)\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate, basename_index\n    if basename_index is None:\n        basename_index = {\n            path.name: path\n            for path in data_root.rglob("*")\n            if path.is_file() and path.suffix.casefold() in {".jpg", ".jpeg", ".png"}\n        }\n    return basename_index.get(Path(normalized).name), basename_index\n\n\ndef load_samples(data_root: Path) -> list[Sample]:\n    """Load the official ``path transcription`` manifest defensively."""\n    label_files = sorted(data_root.rglob("train.txt"))\n    if not label_files:\n        label_files = sorted(\n            path\n            for path in data_root.rglob("*.txt")\n            if "train" in path.name.casefold()\n        )\n    if not label_files:\n        raise FileNotFoundError(f"No train.txt was found below {data_root}")\n    preferred = sorted(\n        label_files,\n        key=lambda path: (\n            "gur" not in path.as_posix().casefold() and "pun" not in path.as_posix().casefold(),\n            len(path.parts),\n        ),\n    )[0]\n    samples: list[Sample] = []\n    basename_index: dict[str, Path] | None = None\n    for raw_line in preferred.read_text(encoding="utf-8-sig").splitlines():\n        parts = raw_line.strip().split(maxsplit=1)\n        if len(parts) != 2:\n            continue\n        image_path, basename_index = _resolve_image(\n            parts[0], preferred, data_root, basename_index\n        )\n        text = normalize_source_text(parts[1])\n        if image_path and text and gurmukhi_ratio(text) >= 0.70:\n            samples.append(Sample(image_path, text))\n    if len(samples) < 1_000:\n        raise ValueError(\n            f"Only {len(samples)} valid Gurmukhi samples were found; refusing to train"\n        )\n    return samples\n\n\ndef split_samples(\n    samples: Sequence[Sample], validation_fraction: float, seed: int\n) -> tuple[list[Sample], list[Sample]]:\n    shuffled = list(samples)\n    random.Random(seed).shuffle(shuffled)\n    validation_size = max(500, min(3_000, round(len(shuffled) * validation_fraction)))\n    return shuffled[validation_size:], shuffled[:validation_size]\n\n\ndef _trim_word(image: Image.Image) -> Image.Image:\n    gray = ImageOps.autocontrast(image.convert("L"))\n    inverted = ImageOps.invert(gray)\n    bbox = inverted.point(lambda value: 255 if value > 18 else 0).getbbox()\n    return gray.crop(bbox) if bbox else gray\n\n\ndef compose_line(\n    samples: Sequence[Sample],\n    indices: Sequence[int],\n    rng: random.Random,\n    *,\n    augment: bool,\n) -> tuple[Image.Image, str]:\n    words: list[Image.Image] = []\n    labels: list[str] = []\n    target_height = rng.randint(52, 72)\n    for index in indices:\n        sample = samples[index]\n        with Image.open(sample.image_path) as source:\n            word = _trim_word(source)\n        scale = target_height / max(1, word.height)\n        width = max(8, round(word.width * scale))\n        word = word.resize((width, target_height), Image.Resampling.LANCZOS)\n        words.append(word)\n        labels.append(sample.text)\n    gaps = [rng.randint(12, 30) for _ in range(max(0, len(words) - 1))]\n    margin = 16\n    canvas_width = sum(word.width for word in words) + sum(gaps) + 2 * margin\n    canvas = Image.new("L", (max(64, canvas_width), target_height + 2 * margin), 255)\n    x = margin\n    for position, word in enumerate(words):\n        y = margin + rng.randint(-3, 3)\n        canvas.paste(word, (x, y))\n        x += word.width + (gaps[position] if position < len(gaps) else 0)\n    if augment:\n        canvas = ImageEnhance.Contrast(canvas).enhance(rng.uniform(0.78, 1.22))\n        canvas = canvas.rotate(\n            rng.uniform(-1.5, 1.5),\n            resample=Image.Resampling.BICUBIC,\n            expand=True,\n            fillcolor=255,\n        )\n    return canvas.convert("RGB"), " ".join(labels)\n\n\ndef levenshtein(left: str, right: str) -> int:\n    previous = list(range(len(right) + 1))\n    for left_index, left_character in enumerate(left, start=1):\n        current = [left_index]\n        for right_index, right_character in enumerate(right, start=1):\n            current.append(\n                min(\n                    current[-1] + 1,\n                    previous[right_index] + 1,\n                    previous[right_index - 1] + (left_character != right_character),\n                )\n            )\n        previous = current\n    return previous[-1]\n\n\ndef corpus_character_error_rate(predictions: Iterable[str], references: Iterable[str]) -> float:\n    errors = 0\n    characters = 0\n    for prediction, reference in zip(predictions, references, strict=True):\n        errors += levenshtein(prediction, reference)\n        characters += len(reference)\n    return errors / max(1, characters)\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef latest_checkpoint(directory: Path) -> Path | None:\n    candidates = [path for path in directory.glob("checkpoint-*") if path.is_dir()]\n    if not candidates:\n        return None\n    return max(candidates, key=lambda path: int(path.name.rsplit("-", 1)[-1]))\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--work-dir", type=Path, default=Path("/kaggle/working/gurmukhi_htr"))\n    parser.add_argument("--dataset-dir", type=Path)\n    parser.add_argument("--base-model", default=BASE_MODEL)\n    parser.add_argument("--seed", type=int, default=20260827)\n    parser.add_argument("--train-examples", type=int, default=90_000)\n    parser.add_argument("--validation-examples", type=int, default=1_000)\n    parser.add_argument("--max-steps", type=int, default=6_000)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--gradient-accumulation", type=int, default=4)\n    parser.add_argument("--learning-rate", type=float, default=5e-5)\n    parser.add_argument("--save-steps", type=int, default=500)\n    parser.add_argument("--max-label-length", type=int, default=96)\n    parser.add_argument("--validation-fraction", type=float, default=0.05)\n    parser.add_argument("--max-validation-cer", type=float, default=0.35)\n    parser.add_argument("--min-script-purity", type=float, default=0.90)\n    parser.add_argument("--min-nonempty-rate", type=float, default=0.95)\n    parser.add_argument("--resume-from", type=Path)\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    args.work_dir.mkdir(parents=True, exist_ok=True)\n    archive = args.work_dir / "downloads" / "gurumukhi.zip"\n    data_root = args.dataset_dir or args.work_dir / "data"\n    if args.dataset_dir is None:\n        download_with_resume(TRAINING_ARCHIVE_URL, archive)\n        marker = data_root / ".extracted"\n        if not marker.exists():\n            safe_extract(archive, data_root)\n            marker.write_text("ok\\n", encoding="utf-8")\n\n    import numpy as np\n    import torch\n    from torch.utils.data import Dataset\n    from transformers import (\n        Seq2SeqTrainer,\n        Seq2SeqTrainingArguments,\n        TrOCRProcessor,\n        VisionEncoderDecoderModel,\n        default_data_collator,\n        set_seed,\n    )\n\n    set_seed(args.seed)\n    samples = load_samples(data_root)\n    train_samples, validation_samples = split_samples(\n        samples, args.validation_fraction, args.seed\n    )\n    processor = TrOCRProcessor.from_pretrained(args.base_model)\n    model = VisionEncoderDecoderModel.from_pretrained(args.base_model)\n    tokenizer = processor.tokenizer\n    model.config.decoder_start_token_id = tokenizer.cls_token_id or tokenizer.bos_token_id\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.eos_token_id = tokenizer.sep_token_id or tokenizer.eos_token_id\n    model.config.max_length = args.max_label_length\n    model.config.num_beams = 4\n    model.config.no_repeat_ngram_size = 0\n    model.config.early_stopping = True\n\n    class LineDataset(Dataset[Any]):\n        def __init__(\n            self,\n            source: Sequence[Sample],\n            length: int,\n            seed: int,\n            augment: bool,\n        ) -> None:\n            self.source = source\n            self.length = length\n            self.seed = seed\n            self.augment = augment\n\n        def __len__(self) -> int:\n            return self.length\n\n        def __getitem__(self, index: int) -> dict[str, torch.Tensor]:\n            rng = random.Random(self.seed + index * 104_729)\n            word_count = 1 if rng.random() < 0.20 else rng.randint(2, 5)\n            indices = [rng.randrange(len(self.source)) for _ in range(word_count)]\n            image, text = compose_line(self.source, indices, rng, augment=self.augment)\n            pixel_values = processor(images=image, return_tensors="pt").pixel_values[0]\n            token_ids = tokenizer(\n                text,\n                padding="max_length",\n                max_length=args.max_label_length,\n                truncation=True,\n            ).input_ids\n            labels = torch.tensor(\n                [token if token != tokenizer.pad_token_id else -100 for token in token_ids],\n                dtype=torch.long,\n            )\n            return {"pixel_values": pixel_values, "labels": labels}\n\n    train_dataset = LineDataset(\n        train_samples, args.train_examples, args.seed, augment=True\n    )\n    validation_dataset = LineDataset(\n        validation_samples,\n        min(args.validation_examples, max(500, len(validation_samples))),\n        args.seed + 9_000_001,\n        augment=False,\n    )\n\n    def decode_metrics(prediction_output: Any) -> dict[str, float]:\n        prediction_ids = prediction_output.predictions\n        if isinstance(prediction_ids, tuple):\n            prediction_ids = prediction_ids[0]\n        label_ids = np.array(prediction_output.label_ids)\n        label_ids[label_ids == -100] = tokenizer.pad_token_id\n        predicted_text = [\n            normalize_source_text(value)\n            for value in tokenizer.batch_decode(prediction_ids, skip_special_tokens=True)\n        ]\n        reference_text = [\n            normalize_source_text(value)\n            for value in tokenizer.batch_decode(label_ids, skip_special_tokens=True)\n        ]\n        nonempty = [value for value in predicted_text if value]\n        return {\n            "cer": corpus_character_error_rate(predicted_text, reference_text),\n            "script_purity": (\n                sum(gurmukhi_ratio(value) for value in nonempty) / max(1, len(nonempty))\n            ),\n            "nonempty_rate": len(nonempty) / max(1, len(predicted_text)),\n        }\n\n    checkpoints = args.work_dir / "checkpoints"\n    training_kwargs: dict[str, Any] = {\n        "output_dir": str(checkpoints),\n        "per_device_train_batch_size": args.batch_size,\n        "per_device_eval_batch_size": args.batch_size,\n        "gradient_accumulation_steps": args.gradient_accumulation,\n        "learning_rate": args.learning_rate,\n        "max_steps": args.max_steps,\n        "warmup_ratio": 0.05,\n        "logging_steps": 50,\n        "save_steps": args.save_steps,\n        "eval_steps": args.save_steps,\n        "save_strategy": "steps",\n        "predict_with_generate": True,\n        "generation_max_length": args.max_label_length,\n        "generation_num_beams": 4,\n        "load_best_model_at_end": True,\n        "metric_for_best_model": "cer",\n        "greater_is_better": False,\n        "save_total_limit": 3,\n        "remove_unused_columns": False,\n        "fp16": bool(torch.cuda.is_available()),\n        "dataloader_num_workers": 2,\n        "report_to": [],\n        "seed": args.seed,\n    }\n    argument_names = inspect.signature(Seq2SeqTrainingArguments).parameters\n    training_kwargs[\n        "eval_strategy" if "eval_strategy" in argument_names else "evaluation_strategy"\n    ] = "steps"\n    training_args = Seq2SeqTrainingArguments(**training_kwargs)\n    trainer = Seq2SeqTrainer(\n        model=model,\n        args=training_args,\n        train_dataset=train_dataset,\n        eval_dataset=validation_dataset,\n        data_collator=default_data_collator,\n        compute_metrics=decode_metrics,\n    )\n    resume = args.resume_from or latest_checkpoint(checkpoints)\n    trainer.train(resume_from_checkpoint=str(resume) if resume else None)\n    prediction = trainer.predict(validation_dataset)\n    metrics = {\n        key.removeprefix("test_"): float(value)\n        for key, value in prediction.metrics.items()\n        if isinstance(value, (int, float))\n    }\n    passed = bool(\n        metrics.get("cer", 1.0) <= args.max_validation_cer\n        and metrics.get("script_purity", 0.0) >= args.min_script_purity\n        and metrics.get("nonempty_rate", 0.0) >= args.min_nonempty_rate\n    )\n\n    artifact = args.work_dir / "artifact" / "gurmukhi_htr"\n    artifact.mkdir(parents=True, exist_ok=True)\n    trainer.model.save_pretrained(artifact, safe_serialization=True)\n    processor.save_pretrained(artifact)\n    report = {\n        "schema_version": "1.0",\n        "passed": passed,\n        "metrics": metrics,\n        "gates": {\n            "max_cer": args.max_validation_cer,\n            "min_script_purity": args.min_script_purity,\n            "min_nonempty_rate": args.min_nonempty_rate,\n        },\n        "validation_examples": len(validation_dataset),\n        "word_level_source_samples": len(samples),\n        "synthetic_line_validation": True,\n        "domain_warning": (\n            "The public corpus is word-level; synthetic lines do not prove accuracy on "\n            "historical legal or medical pages. Runtime source validation remains mandatory."\n        ),\n    }\n    (artifact / "validation_report.json").write_text(\n        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    manifest = {\n        "schema_version": "1.0",\n        "provider_id": "gurmukhi_htr",\n        "backend": "transformers_vision_encoder_decoder",\n        "model_subdirectory": ".",\n        "supported_languages": ["pa"],\n        "supported_scripts": ["gurmukhi"],\n        "confidence_capability": "sequence_probability",\n        "source_language_output_only": True,\n        "handwriting_validated": passed,\n        "validation_report": "validation_report.json",\n        "base_model": args.base_model,\n        "training_dataset": {\n            "name": "IIIT-INDIC-HW-WORDS Gurumukhi",\n            "url": TRAINING_ARCHIVE_URL,\n            "source_level": "word",\n        },\n        "training_arguments": {\n            key: value\n            for key, value in vars(args).items()\n            if key not in {"dataset_dir", "resume_from"}\n            and isinstance(value, (str, int, float, bool, type(None)))\n        },\n    }\n    (artifact / "htr_manifest.json").write_text(\n        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    checksums = {\n        path.name: sha256(path)\n        for path in artifact.iterdir()\n        if path.is_file() and path.name != "checksums.json"\n    }\n    (artifact / "checksums.json").write_text(\n        json.dumps(checksums, indent=2), encoding="utf-8"\n    )\n    archive_path = shutil.make_archive(\n        str(args.work_dir / "gurmukhi_htr_model"), "zip", artifact\n    )\n    print(json.dumps({"artifact": archive_path, "validation": report}, indent=2))\n    if not passed:\n        raise SystemExit(\n            "Training completed, but validation gates failed. The bundle was exported "\n            "for diagnosis and will not be auto-routed by the application."\n        )\n\n\nif __name__ == "__main__":\n    main()\n'
script_path = Path('/kaggle/working/gurmukhi_train.py')
script_path.write_text(training_source, encoding='utf-8')
print(f'Prepared {script_path}')


In [ ]:
import subprocess, sys
from pathlib import Path
checkpoint_candidates = sorted(
    Path('/kaggle/input').rglob('checkpoint-*'),
    key=lambda p: int(p.name.rsplit('-', 1)[-1]) if p.name.rsplit('-', 1)[-1].isdigit() else -1,
)
command = [sys.executable, '/kaggle/working/gurmukhi_train.py']
if checkpoint_candidates:
    command += ['--resume-from', str(checkpoint_candidates[-1])]
print('Starting/resuming source-language HTR training')
subprocess.run(command, check=True)
